In [15]:
from pydantic_ai import Agent, RunContext
import asyncio
from pydantic import BaseModel
import httpx
import requests

In [ ]:
import os
api_key = os.getenv("GOOGLE_API_KEY")
print(api_key)

## Agente

In [ ]:
# result = agent.run_sync("What is Pydantic AI?")  # "Ejecuta la petición y bloquea el programa hasta que llegue la respuesta." 
# Falla porque Jupyter ya está usando internamente un event loop de asyncio y run_sync() intenta arrancar otro evento loop.
  # "Pide la respuesta y cuando llegue, continúa." Evenloop gestiona la espera.

agent = Agent(
    "google:gemini-2.5-flash",
    instructions="Answer in English",
)

result = await agent.run("¿Cuál es la capital de España?")

print(result.output)


The capital of Spain is **Madrid**.


In [ ]:
# Cuando se tienen muchos documentos

# async def extract_document(
#     agent: Agent,
#     row: pd.Series,
# ) -> dict:
#     result = await agent.run(row["texto_limpio"])
#     return result.output.model_dump()

# tasks = [
#     extract_document(agent, row)
#     for _, row in pending.iterrows()
# ]

# records = await asyncio.gather(*tasks)

## Output estructurado

In [10]:
class CityInfo(BaseModel):
    name: str
    country: str
    population: int
    fun_fact: str


agent = Agent(
    "google:gemini-2.5-flash",
    output_type=CityInfo,
)

result = await agent.run("Tell me about Tokyo")

print(result.output)
print(f"{result.output.name}, {result.output.country}")
print(f"Population: {result.output.population:,}")
print(f"Fun fact: {result.output.fun_fact}")

name='Tokyo' country='Japan' population=13960000 fun_fact='Tokyo has the most Michelin-starred restaurants of any city in the world.'
Tokyo, Japan
Population: 13,960,000
Fun fact: Tokyo has the most Michelin-starred restaurants of any city in the world.


## Tools

In [ ]:
# Creamos un agente que utilizará Gemini.
# Las instrucciones actúan como un "prompt de sistema":
# definen el comportamiento general del modelo.
agent = Agent(
    "google:gemini-2.5-flash",
    instructions="Help users with cat breeds. Be concise.",
)


# Registramos una herramienta (tool) que el agente puede utilizar.
#
# El modelo conoce esta herramienta porque PydanticAI le envía:
# - el nombre de la función (find_breed_info)
# - el docstring
# - los parámetros y sus tipos
#
# Cuando el modelo necesite información sobre una raza de gato,
# podrá llamar automáticamente a esta función.
#
# @agent.tool_plain indica que la función no necesita contexto
# adicional del agente.
@agent.tool_plain
async def find_breed_info(breed_name: str) -> dict:
    """
    Busca información sobre una raza de gato utilizando The Cat API.
    """

    # Creamos un cliente HTTP asíncrono.
    # 'async with' garantiza que la conexión se cierre correctamente
    # al finalizar la petición.
    async with httpx.AsyncClient() as client:

        # Realizamos una petición GET a la API pública.
        response = await client.get(
            "https://api.thecatapi.com/v1/breeds"
        )

    # Lanza una excepción si la respuesta HTTP contiene errores
    # (404, 500, etc.).
    response.raise_for_status()

    # Convertimos la respuesta JSON en una lista de diccionarios Python.
    breeds = response.json()

    # Recorremos todas las razas devueltas por la API.
    for breed in breeds:

        # Si encontramos la raza solicitada,
        # devolvemos toda su información.
        if breed["name"] == breed_name:
            return breed

    # Si no se encuentra la raza, devolvemos un mensaje de error.
    return {"error": "Breed not found"}


# Enviamos una consulta al agente.
#
# El modelo analizará la pregunta y decidirá si necesita llamar
# a la herramienta find_breed_info().
result = await agent.run(
    "Tell me about Siamese cats."
)

# Mostramos la respuesta final generada por el agente.
print(result.output)

Siamese cats are known for being active, agile, clever, sociable, loving, and energetic. They are very fond of their owners, will follow you around, and are quite talkative and opinionated. They are a demanding and social breed that doesn't like to be left alone for long periods. They originated in Thailand and have a life span of 12-15 years.


## Obtaining data from database

In [ ]:
class UserDatabase:
    """
    Simula una base de datos de usuarios.

    En este ejemplo los datos se obtienen de la API pública
    JSONPlaceholder.
    """
    _base_url = "https://jsonplaceholder.typicode.com"

    def get_user_info(self, user_id: int) -> dict:
        """
        Recupera la información de un usuario.
        """
        response = requests.get(
            f"{self._base_url}/users/{user_id}"
        )

        response.raise_for_status()

        print(response.json())
        return response.json()


class UserSummary(BaseModel):
    """
    Estructura de salida que debe devolver el agente.

    Pydantic validará automáticamente que la respuesta
    contenga estos campos.
    """
    name: str
    email: str
    company: str
    address: str


# Creamos el agente.
#
# output_type:
#   Indica que la respuesta final debe ajustarse al modelo
#   UserSummary.
#
# deps_type:
#   Define el tipo de dependencia que se inyectará en el
#   agente durante la ejecución.
agent = Agent(
    "google:gemini-2.5-flash",
    output_type=UserSummary,
    deps_type=UserDatabase,
    instructions=(
        "You retrieve user information from an external database. "
        "Use the available tools to gather user info, "
        "then return a structured summary."
    ),
)


# Esta función se expone al modelo como una herramienta.
#
# El LLM conoce esta herramienta porque PydanticAI le envía:
# - el nombre de la función
# - el docstring
# - los parámetros
#
# Cuando necesite datos de un usuario podrá llamarla
# automáticamente.
@agent.tool
def fetch_user(
    ctx: RunContext[UserDatabase],
    user_id: int,
) -> str:
    """
    Fetch user profile from the service.
    """

    try:
        # Accedemos a la dependencia inyectada.
        # ctx.deps contiene una instancia de UserDatabase.
        user = ctx.deps.get_user_info(user_id)

        return str(user)

    except requests.HTTPError:
        return f"User with ID {user_id} not found"


# Creamos la dependencia que será utilizada por la herramienta.
db = UserDatabase()


# En notebooks usamos await en lugar de run_sync().
result = await agent.run(
    "Get a summary for user 7",
    deps=db,
)


# result.output ya es una instancia validada de UserSummary.
print(f"Name: {result.output.name}")
print(f"Email: {result.output.email}")
print(f"Company: {result.output.company}")
print(f"Adress: {result.output.address}")

{'id': 7, 'name': 'Kurtis Weissnat', 'username': 'Elwyn.Skiles', 'email': 'Telly.Hoeger@billy.biz', 'address': {'street': 'Rex Trail', 'suite': 'Suite 280', 'city': 'Howemouth', 'zipcode': '58804-1099', 'geo': {'lat': '24.8918', 'lng': '21.8984'}}, 'phone': '210.067.6132', 'website': 'elvis.io', 'company': {'name': 'Johns Group', 'catchPhrase': 'Configurable multimedia task-force', 'bs': 'generate enterprise e-tailers'}}
Name: Kurtis Weissnat
Email: Telly.Hoeger@billy.biz
Company: Johns Group
Adress: Rex Trail, Suite 280, Howemouth, 58804-1099
